### Công thức bayes

Công thức:

$$\boxed{P(A|B)=\frac{P(B|A)\times P(A)}{P(B)}}$$

ví dụ: 
$$P(Spam|Lottery)=
\frac{P(Lottery|Spam)\times P(Spam)}
{P(Lottery)}$$  
\(P(Spam)\): Xác suất một email bất kỳ là Spam, gọi là Prior.



\(P(Lottery|Spam)\): Trong các email Spam, bao nhiêu email chứa từ lottery.



\(P(Lottery)\): Xác suất email bất kỳ chứa lottery.



\(P(Spam|Lottery)\): Biết email chứa lottery, xác suất nó là Spam bao nhiêu, gọi là Posterior.  

- Ví dụ như 1 email mà nó có cả 2 từ là sale,lottery thì 
$$P(Lottery,Sale|Spam)
\approx
P(Lottery|Spam)\times P(Sale|Spam)$$

-
3. Công thức Naive Bayes dùng để Classification
Với email chứa các từ (x1,x2,...xn)
$$
\boxed{
P(c|X)\propto P(c)\prod_{i=1}^{n}P(x_i|c)
}
$$

Trong đó:
- \(c\): class, chẳng hạn Spam hoặc Ham.
- \(P(c)\): Prior của class.
- \(P(x_i|c)\): Xác suất xuất hiện từ \(x_i\) khi biết class \(c\).
- \(\prod\): Ký hiệu nhân tất cả các xác suất.
Ta tính cho từng class rồi chọn class có kết quả cao nhất.

In [4]:
import numpy as np

import pandas as pd
emails = pd.read_csv('emails.csv')
emails[:10]


,text,spam
0,Subject: naturally irresistible your corporate...,1
1,Subject: the stock trading gunslinger fanny i...,1
2,Subject: unbelievable new homes made easy im ...,1
3,Subject: 4 color printing special request add...,1
4,"Subject: do not have money , get software cds ...",1
5,"Subject: great nnews hello , welcome to medzo...",1
6,Subject: here ' s a hot play in motion homela...,1
7,Subject: save your money buy getting this thin...,1
8,Subject: undeliverable : home based business f...,1
9,Subject: save your money buy getting this thin...,1


In [5]:
def process_email(text):
    text = text.lower()
    return list(set(text.split()))

emails['words'] = emails['text'].apply(process_email)

In [6]:
emails[:10]

,text,spam,words
0,Subject: naturally irresistible your corporate...,1,"[website, but, list, task, content, is, you, e..."
1,Subject: the stock trading gunslinger fanny i...,1,"[palfrey, perspicuous, nameable, stock, gunsli..."
2,Subject: unbelievable new homes made easy im ...,1,"[time, our, unconditionally, advantage, websit..."
3,Subject: 4 color printing special request add...,1,"[our, version, phone, ramsey, or, 626, &, form..."
4,"Subject: do not have money , get software cds ...",1,"[?, money, are, cds, grow, by, is, ended, yet,..."
5,"Subject: great nnews hello , welcome to medzo...",1,"[%, phar, ourselves, over, 75, nice, groundsel..."
6,Subject: here ' s a hot play in motion homela...,1,"[ourselves, predictions, projections, does, pr..."
7,Subject: save your money buy getting this thin...,1,"[tried, has, ?, over, money, effect, are, cial..."
8,Subject: undeliverable : home based business f...,1,"[75, sun, jan, l, is, q, feb, tfi, recipient, ..."
9,Subject: save your money buy getting this thin...,1,"[tried, has, ?, over, money, effect, aicohoi, ..."


In [7]:
num_emails = len(emails)
num_spam = sum(emails['spam'])

print("Number of emails:", num_emails)
print("Number of spam emails:", num_spam)
print()

# Calculating the prior probability that an email is spam
print("Probability of spam:", num_spam/num_emails)

Number of emails: 5728
Number of spam emails: 1368

Probability of spam: 0.2388268156424581


In [8]:
model = {}

# Training process
for index, email in emails.iterrows():
    for word in email['words']:
        if word not in model:
            model[word] = {'spam': 1, 'ham': 1}
        if word in model:
            if email['spam']:
                model[word]['spam'] += 1
            else:
                model[word]['ham'] += 1

In [9]:
model['lottery']

{'spam': 9, 'ham': 1}

In [10]:
model['sale']

{'spam': 39, 'ham': 42}

In [11]:
def predict_bayes(word):
    word = word.lower()
    num_spam_with_word = model[word]['spam']
    num_ham_with_word = model[word]['ham']
    return 1.0*num_spam_with_word/(num_spam_with_word + num_ham_with_word)

In [12]:
predict_bayes('lottery')

0.9

In [13]:
predict_bayes('sale')

0.48148148148148145

In [21]:
def predict_naive_bayes(email):
    total = len(emails)
    num_spam = sum(emails['spam'])
    num_ham = total - num_spam
    email = email.lower()
    words = set(email.split())
    spams = [1.0]
    hams = [1.0]
    for word in words:
        if word in model:
            spams.append(model[word]['spam']/num_spam*total)
            hams.append(model[word]['ham']/num_ham*total)
    prod_spams = np.compat.long(np.prod(spams)*num_spam)
    prod_hams = np.compat.long(np.prod(hams)*num_ham)
    return prod_spams/(prod_spams + prod_hams)